
# Burrows–Wheeler Transform (BWT) & FM-Index:

If you’ve ever wondered how bioinformatics tools like **Bowtie2** or **BWA** can search through the entire human genome (3.2 billion DNA letters!) on a regular laptop without it catching fire, the answer is the **Burrows–Wheeler Transform (BWT)**.

Normally, if you want to find a short DNA read inside a massive genome, scanning from start to finish takes $O(N)$ time—which is way too slow when you have millions of reads. BWT solves this by scrambling a sequence in a very specific, **100% reversible** way so that:
1. **Identical letters group together into "runs"** (like `AAAA` or `TTTT`), making the genome super easy to compress.
2. **Repeated patterns line up alphabetically**, allowing us to search for any pattern of length $M$ in just **$O(M)$ steps**—no matter how huge the genome is!

---

## How the Code Works (Step-by-Step)

### 1. Input Validation & Auto-Detection (`validate_raw_sequence` & `validate_bwt_input`)
Before doing any math, the code acts like a bouncer at the door to make sure the input won't break the algorithm:

* **Auto-Detecting the Sequence Type:**
  * If the string only uses `A, T, G, C, N`, it’s flagged as **DNA**.
  * If it has `U` instead of `T` (`A, U, G, C, N`), it’s **RNA**.
  * If it uses standard amino acid letters (like `M, K, W, V, F`), it’s **PROTEIN**.
  * Anything else (like `"BANANA"`—which is legally required in every BWT assignment ever written) is treated as **TEXT**.
* **Protecting the Sentinel (`$`):**
  * BWT needs a special "end-of-string" marker called the **sentinel (`$`)**. We must make sure the user didn't sneak a `$` into their raw sequence.
* **Fixing the ASCII Betrayal (`_sentinel_sort_key`):**
  * By rule, `$` **must** be alphabetically smaller than every other character. In Python’s standard ASCII table, a space (`' '`, ASCII 32) is actually smaller than `'$'` (ASCII 36). Our helper function `_sentinel_sort_key` forces `$` to have priority `0` so it *always* sorts to the very top row, no matter what characters are in your string.

---

### 2. Forward BWT — Scrambling the Sequence (`bwt_encode`)
How do we encode a sequence like `GATTACA` or `BANANA`? Let's trace `BANANA`:

1. **Slap the Sentinel (`$`) on the end:**  
   `BANANA` $\rightarrow$ `BANANA$` (Total length $N = 7$).
2. **Generate all $N$ Cyclic Rotations:**  
   We pluck letters off the front and move them to the back, one by one, until we’ve gone full circle. Notice how in every single row, **the Last letter (`L`) is actually the letter that came right BEFORE the First letter (`F`) in the original word!**
3. **Sort the Rotations Alphabetically (with `$` first):**  
   Sorting all the rotations groups identical suffixes ("right-contexts") together:

   | Row | Shift (`SA[i]`) | First (`F`) | Sorted Rotation Matrix | Last (`L` / BWT) | Note |
   | :-: | :-: | :-: | :--- | :-: | :--- |
   | **0** | 6 | `$` | `$BANANA` | **`A`** | |
   | **1** | 5 | `A` | `A$BANAN` | **`N`** | |
   | **2** | 3 | `A` | `ANA$BAN` | **`N`** | |
   | **3** | 1 | `A` | `ANANA$B` | **`B`** | |
   | **4** | 0 | `B` | `BANANA$` | **`$`** | **$\leftarrow$ Primary Index (Row 4)** |
   | **5** | 4 | `N` | `NA$BANA` | **`A`** | |
   | **6** | 2 | `N` | `NANA$BA` | **`A`** | |

4. **Extract the Outputs:**
   * **BWT String (`L` column):** Reading the last column top-to-bottom gives **`ANNB$AA`**. Look at how the `N`s and `A`s clumped together!
   * **Primary Index (`4`):** The row where our original, unshifted string (`BANANA$`, shift 0) ended up. Because the original string ends with `$`, `L[primary_index]` is *always* where `$` sits!
   * **Suffix Array (`SA = [6, 5, 3, 1, 0, 4, 2]`):** Just a list of how many characters each sorted row was shifted from the original string. We save this for genome coordinate lookups later.

---

### 3. Inverse BWT (`bwt_decode` & `build_lf_mapping_structures`)
Here is the wildest part of BWT: if someone deletes the entire rotation matrix and *only* hands you the BWT string **`ANNB$AA`** and the Primary Index **`4`**, how do you get `BANANA` back without rebuilding the $N \times N$ grid?

We use three clever pieces of information:

1. **Recovering the First Column (`F`):**  
   Since `F` is just all the characters of the sequence sorted alphabetically, we get `F` simply by sorting `L` (`ANNB$AA` sorted $\rightarrow$ `$AAABNN`).
2. **The C-Table (`C[char]`):**  
   `C[char]` counts how many total characters in the string are strictly smaller than `char`.  
   *Why do we care?* Because in a 0-based table, that count is the **exact row number where `char` starts in column `F`!**
   * For `BANANA$`: `$` starts at row `0` (`C['$'] = 0`), `A` starts at row `1` (`C['A'] = 1`), `B` starts at row `4` (`C['B'] = 4`), and `N` starts at row `5` (`C['N'] = 5`).
3. **The LF-Mapping Property (Last-to-First):**  
   In both `F` and `L`, multiple `'A'`s appear. How do we know *which* `'A'` in `L` matches *which* `'A'` in `F`?  
   * **Rule:** The **1st `'A'`** in `L` is the exact same text character as the **1st `'A'`** in `F`. The **2nd `'A'`** in `L` is the **2nd `'A'`** in `F`, and so on!  
   * *Why?* Because all rows starting with `'A'` in `F` are sorted by the letters that come *after* `'A'`. And all rows ending with `'A'` in `L` are *also* sorted by the exact same letters coming *after* `'A'`! Since both groups use the exact same tie-breaker, their relative order (`rank`) stays identical.

#### The Reconstruction Formula:
$$\text{LF}[i] = C[\,L[i]\,] + \text{rank}_0(L[i])$$
*(Translation: To find where the letter at `L[i]` lives in column `F`, take the starting row of that letter in `F` (`C[L[i]]`) and add how many times you've already seen that letter in `L` above row `i`.)*

* **Walking Backwards in $O(N)$ Steps:**
  1. We start at Row `0` (where `F[0] = '$'`, which is the end of our word).
  2. The letter in `L[0]` is `'A'` ( specifically the 1st `'A'` in `L`, so `rank = 0`). That means `'A'` comes right before `'$'`!
  3. We jump to `LF[0] = C['A'] + 0 = 1 + 0 = Row 1`.
  4. At Row `1`, `L[1]` is `'N'` (1st `'N'` in `L`, `rank = 0`), so `'N'` comes before `'A'`. We jump to `LF[1] = C['N'] + 0 = 5`.
  5. Repeat $N$ times until we hit `'$'` again! That spells `ANANAB` backwards, which we flip to get **`BANANA`**.

---

### 4. Biological Analysis & Repeat Clustering (`analyze_biological_sequence`)
Why do biologists care so much about BWT? This function bridges the computer science with real-world genomics:

* **GC / AT / AU Content:**
  * In DNA and RNA, **Guanine (`G`)** and **Cytosine (`C`)** form **3 hydrogen bonds** with each other, while **Adenine (`A`)** and **Thymine (`T`) / Uracil (`U`)** only form **2**.
  * A higher **GC%**\ means the DNA is much more stable and requires a higher temperature ($T_m$) to melt apart during lab techniques like PCR.
* **Homopolymers & Repeated $k$-mers (`find_repetitive_patterns`):**
  * Genomes are full of repeats—both **homopolymers** (like `AAAAA`) and **repeated $k$-mers** (short words of length 2, 3, or 4 like `ATG` that appear multiple times).
* **Run-Length Encoding (`count_runs`):**
  * When a motif like `ANA` repeats in a sequence, every rotation starting with `NA...` gets sorted into adjacent rows in the BWT matrix. That forces the letters right before `NA` (the `'A'`s) to sit next to each other in the BWT output!
  * `count_runs` compresses consecutive identical letters into counts (e.g., `BANANA` has **6 runs**: `B1 A1 N1 A1 N1 A1`, whereas its BWT `ANNB$AA` has fewer runs because the `N`s and `A`s grouped up: `A1 N2 B1 $1 A2`). On a real genome, this shrinks file sizes dramatically.

---

### 5. FM-Index Backward Search (`fm_index_backward_search`)
Instead of decoding the whole genome just to find where a short DNA read (pattern $P$, like `"ANA"`) is located, the **FM-Index** searches the compressed BWT directly by reading the pattern **backwards** (right-to-left):

1. **Keep an Occurrence Checkpoint Table (`Occ(char, row)`):**  
   This table simply records how many times `char` has appeared in the BWT column `L` from row `0` down to `row`.
2. **Narrow the Row Range `[start_row .. end_row]`:**  
   We start with all rows (`0` to $N-1$) as candidates, and read our pattern `"ANA"` from right to left (`'A'`, then `'N'`, then `'A'`):
   * At each letter `c`, we update our row range using the exact same LF-mapping logic:
     $$\text{new\_start} = C[c] + \text{Occ}(c, \text{start\_row} - 1)$$
     $$\text{new\_end} = C[c] + \text{Occ}(c, \text{end\_row}) - 1$$
   * **Step 1 (`'A'`):** Narrows `[0..6]` down to rows `[1..3]` (all rows starting with `'A'`).
   * **Step 2 (`'N'`):** Checks how many `'N'`s appear in `L[1..3]` (there are two!) and jumps to their rows in `F`: `[5..6]` (rows starting with `"NA"`).
   * **Step 3 (`'A'`):** Checks how many `'A'`s appear in `L[5..6]` (two!) and jumps to their rows in `F`: **`[2..3]`** (rows starting with `"ANA"`).
3. **Read the Genomic Coordinates from the Suffix Array (`SA`):**  
   Our final row range is `[2..3]`. Looking up `SA[2]` and `SA[3]` gives **`[3, 1]`**—meaning `"ANA"` appears at **index 1** and **index 3** in `"BANANA"`! And we found both matches in just 3 tiny math steps.

---              
## Implementation & AI Usage Note

### Implementation Note
* The core algorithmic workflows in this project—including cyclic rotation sorting, $C$-table construction, Last-to-First (LF) mapping inversion, and FM-Index backward search—were manually designed and implemented using the primary literature and educational coding resources listed in the references (specifically Burrows & Wheeler, 1994; Ferragina & Manzini, 2000; Compeau & Pevzner, 2018; the Rosalind `BA9I–BA9N` problem track; Prof. Ben Langmead’s JHU Python notebooks; and GeeksforGeeks).
* AI assistance was used strictly for post-implementation code tuning and debugging—such as fixing edge-case input validation, handling the custom ASCII sort priority for the `'$'` sentinel, and tidying up the CLI table formatting. All core algorithm design, biological sequence analyses, and test-case verifications were completed manually.

---

## References

### 1. Core Algorithms (BWT, LF-Mapping, Suffix Arrays, & FM-Index)
1. **Burrows, M., & Wheeler, D. J. (1994).** *A block-sorting lossless data compression algorithm.* Digital Equipment Corporation, SRC Research Report 124.  
   *(Original paper introducing the Burrows–Wheeler Transform, cyclic rotations, and the Last-to-First [LF] mapping property used in `bwt_encode` and `bwt_decode`.)*
2. **Ferragina, P., & Manzini, G. (2000).** *Opportunistic data structures with applications.* In Proceedings of the 41st Annual Symposium on Foundations of Computer Science (FOCS) (pp. 390–398). IEEE. https://doi.org/10.1109/SFCS.2000.892127  
   *(Introduced the **FM-Index**, combining the BWT with the $C$-table and Occurrence [$Occ$] checkpoints for $O(|P|)$ backward search used in `fm_index_backward_search`.)*
3. **Manber, U., & Myers, G. (1993).** *Suffix arrays: A new method for on-line string searches.* SIAM Journal on Computing, 22(5), 935–948. https://doi.org/10.1137/0222058  
   *(Introduced the Suffix Array [$SA$] data structure used to recover genomic coordinates from BWT row intervals.)*

### 2. Bioinformatics & Genome Alignment Applications
4. **Li, H., & Durbin, R. (2009).** *Fast and accurate short read alignment with Burrows–Wheeler transform.* Bioinformatics, 25(14), 1754–1760. https://doi.org/10.1093/bioinformatics/btp324  
   *(Paper behind **BWA**, demonstrating how BWT and FM-Index allow ultra-fast mapping of DNA sequencing reads to large reference genomes.)*
5. **Langmead, B., & Salzberg, S. L. (2012).** *Fast gapped-read alignment with Bowtie 2.* Nature Methods, 9(4), 357–359. https://doi.org/10.1038/nmeth.1923  
   *(Describes **Bowtie 2**, combining FM-Index backward search with dynamic programming for gapped DNA read alignment.)*
6. **Kim, D., Paggi, J. M., Park, C., Bennett, C., & Salzberg, S. L. (2019).** *Graph-based genome alignment and genotyping with HISAT2 and HISAT-genotype.* Nature Biotechnology, 37(8), 907–915. https://doi.org/10.1038/s41587-019-0201-4  
   *(Describes **HISAT2**, which uses hierarchical BWT/FM-Index structures for fast RNA-Seq splice-junction alignment.)*

### 3. Textbooks & Documentation
7. **Compeau, P., & Pevzner, P. A. (2018).** *Bioinformatics Algorithms: An Active Learning Approach* (3rd ed., Chapter 9: "How Do We Locate Disease-Causing Mutations?"). Active Learning Publishers. ISBN: 978-0990374633.  
   *(Standard bioinformatics textbook covering BWT run-length clustering, the First-Last property, and pattern matching in genomes.)*

### 4. Web/code references
9. **Langmead, B.** *Computational Genomics Course Notebooks: Burrows–Wheeler Transform, Reversing BWT, and FM Index (Python).* Johns Hopkins University. Available at: https://github.com/BenLangmead/comp-genomics-class
10. **Rosalind Bioinformatics Platform.** *Bioinformatics Textbook Track (Problems BA9I–BA9N: BWT Construction, LF-Mapping, and BWMatching).* Available at: https://rosalind.info/problems/list-view/?location=bioinformatics-textbook-track
11. **GeeksforGeeks.** *Burrows–Wheeler Data Transform Algorithm & Inverting the BWT in Python.* Available at: https://www.geeksforgeeks.org/burrows-wheeler-data-transform-algorithm/

In [ ]:
import sys
from collections import Counter
from typing import Dict, List, Optional, Tuple

# '$' marks the end of the sequence. In BWT, '$' must be smaller than every other letter.
SENTINEL = "$"

DNA_VALID = set("ATGCN")
RNA_VALID = set("AUGCN")
PROTEIN_STANDARD = set("ACDEFGHIKLMNPQRSTVWY")
PROTEIN_VALID = PROTEIN_STANDARD | set("BZXUO")
AA_NAMES = {
    "A": "Alanine", "C": "Cysteine", "D": "Aspartate", "E": "Glutamate",
    "F": "Phenylalanine", "G": "Glycine", "H": "Histidine", "I": "Isoleucine",
    "K": "Lysine", "L": "Leucine", "M": "Methionine", "N": "Asparagine",
    "P": "Proline", "Q": "Glutamine", "R": "Arginine", "S": "Serine",
    "T": "Threonine", "V": "Valine", "W": "Tryptophan", "Y": "Tyrosine",
    "B": "Asx (D/N)", "Z": "Glx (E/Q)", "X": "Unknown", "U": "Selenocysteine", "O": "Pyrrolysine",
}

# 1. INPUT VALIDATION & AUTO-DETECTION

def _sentinel_sort_key(text: str) -> Tuple[Tuple[int, str], ...]:
    """Helper for sorting: gives '$' a priority of 0 so it always comes first."""
    return tuple((0, "") if ch == SENTINEL else (1, ch) for ch in text)


def validate_raw_sequence(sequence: str, seq_type: str = "AUTO") -> Tuple[str, str]:
    """Cleans the input string and checks if it is DNA, RNA, PROTEIN, or TEXT."""
    if not sequence or not sequence.strip():
        raise ValueError("Input sequence cannot be empty.")

    cleaned = sequence.strip()
    if SENTINEL in cleaned:
        raise ValueError(f"Sequence cannot contain '{SENTINEL}' (reserved for end-of-string).")

    seq_type = seq_type.upper().strip()
    upper_seq = cleaned.upper()
    chars = set(upper_seq)

    # Auto-detect what kind of sequence the user typed
    if seq_type == "AUTO":
        if chars.issubset(DNA_VALID) and ("T" in chars or chars.issubset(set("AGCN"))):
            return upper_seq, "DNA"
        if chars.issubset(RNA_VALID) and "U" in chars:
            return upper_seq, "RNA"
        if chars.issubset(PROTEIN_STANDARD) and cleaned.isalpha():
            return upper_seq, "PROTEIN"
        return cleaned, "TEXT"

    # Otherwise, check against the specific type the user asked for
    valid_sets = {"DNA": DNA_VALID, "RNA": RNA_VALID, "PROTEIN": PROTEIN_VALID}
    if seq_type in valid_sets:
        invalid = chars - valid_sets[seq_type]
        if invalid:
            raise ValueError(f"Invalid {seq_type} letters found: {sorted(invalid)}")
        return upper_seq, seq_type
    if seq_type == "TEXT":
        return cleaned, "TEXT"

    raise ValueError(f"Unknown sequence type: '{seq_type}'.")


def validate_bwt_input(bwt_string: str, primary_index: Optional[int] = None) -> Tuple[str, int]:
    """Checks that an encoded BWT string has exactly one '$' and a valid Primary Index."""
    bwt = (bwt_string or "").strip()
    if len(bwt) < 2:
        raise ValueError("BWT string must be at least 2 characters long (sequence + '$').")
    if bwt.count(SENTINEL) != 1:
        raise ValueError(f"BWT string must contain exactly ONE '{SENTINEL}'.")

    # Because the unshifted original string ends with '$', '$' is always at L[primary_index]
    sentinel_pos = bwt.index(SENTINEL)
    if primary_index is None:
        return bwt, sentinel_pos
    if not (0 <= primary_index < len(bwt)):
        raise ValueError(f"Primary index {primary_index} is out of range (0..{len(bwt) - 1}).")
    if bwt[primary_index] != SENTINEL:
        raise ValueError(f"Mismatch: L[{primary_index}] is '{bwt[primary_index]}', but '{SENTINEL}' is at {sentinel_pos}.")
    return bwt, primary_index

# 2. FORWARD BWT (ENCODING: SEQUENCE -> BWT STRING)

def bwt_encode(sequence: str, verbose: bool = True) -> Tuple[str, int]:
    """
    How BWT Encoding works:
    1. Add '$' to the end of the sequence.
    2. Create all cyclic rotations (move first 'i' letters to the back).
    3. Sort all rotations alphabetically (with '$' first).
    4. The Last column (L) is our BWT string!
    """
    cleaned_seq, detected_type = validate_raw_sequence(sequence, "AUTO")
    seq_with_dollar = cleaned_seq + SENTINEL
    n = len(seq_with_dollar)

    # Step 1 & 2: Generate all rotations -> (original_shift_index, rotated_string)
    unsorted_rotations = [
        (i, seq_with_dollar[i:] + seq_with_dollar[:i]) for i in range(n)
    ]

    # Step 3: Sort rotations alphabetically
    sorted_rotations = sorted(unsorted_rotations, key=lambda item: _sentinel_sort_key(item[1]))

    # Step 4: Read off the First column (F), Last column (L / BWT), and Suffix Array (SA)
    f_column = "".join(rot[0] for _, rot in sorted_rotations)
    bwt_string = "".join(rot[-1] for _, rot in sorted_rotations)
    suffix_array = [orig_shift for orig_shift, _ in sorted_rotations]
    primary_index = suffix_array.index(0)  # Row where original unshifted string (shift 0) landed

    if verbose:
        w = max(15, n + 2)
        print(f"\n{'='*75}\nFORWARD BWT ENCODING | Input: {cleaned_seq} ({detected_type}) | T: {seq_with_dollar}\n{'='*75}")
        print("STEP 1: All Cyclic Rotations (Moving letters in a circle)")
        print(f"  {'Shift (0/1-b)':<14} | {'Rotation':<{w}} | {'First':<5} | {'Last'}")
        print("  " + "-" * (w + 32))
        for i, rot in unsorted_rotations:
            note = " <-- Original" if i == 0 else ""
            print(f"  {f'{i} / {i+1}':<14} | {rot:<{w}} | {rot[0]:^5} | {rot[-1]:^4}{note}")

        print("\nSTEP 2: Sorted BWT Matrix (Sorting puts identical contexts next to each other)")
        print(f"  {'Row (0/1-b)':<12} | {'SA[i]':<5} | {'F':<3} | {'Sorted Rotation':<{w}} | {'L (BWT)'}")
        print("  " + "-" * (w + 38))
        for row_idx, (orig_i, rot) in enumerate(sorted_rotations):
            note = " <-- Primary Index" if row_idx == primary_index else ""
            print(f"  {f'{row_idx} / {row_idx+1}':<12} | {orig_i:<5} | {rot[0]:^3} | {rot:<{w}} | {rot[-1]:^7}{note}")

        print(f"\nSTEP 3: Result -> F: {f_column} | BWT (L): {bwt_string} | SA: {suffix_array} | Primary Idx: {primary_index}")
        print("=" * 75)

    return bwt_string, primary_index

# 3. INVERSE BWT (DECODING: BWT STRING -> ORIGINAL SEQUENCE)

def build_lf_mapping_structures(bwt_string: str) -> Dict[str, object]:
    """
    Builds the helper tables needed to reverse the BWT:
    - F column: Just the BWT string sorted alphabetically.
    - C-table (C[char]): How many total letters in the string are smaller than 'char'
                         (this tells us what row 'char' starts at in the F column!).
    - LF Mapping (LF[row]): Tells us where a letter in L appears in F.
                            Formula: LF[row] = C[char] + (how many times 'char' appeared in L before this row).
    """
    l_col = list(bwt_string)
    f_col = sorted(l_col, key=_sentinel_sort_key)
    alphabet = sorted(set(l_col), key=_sentinel_sort_key)
    counts = Counter(l_col)

    # Build C-table: cumulative counts of smaller characters
    c_table, total_so_far = {}, 0
    for ch in alphabet:
        c_table[ch] = total_so_far
        total_so_far += counts[ch]

    # Count occurrence rank of each letter in F (0th 'A', 1st 'A', etc.)
    f_seen, f_rank = Counter(), []
    for ch in f_col:
        f_rank.append(f_seen[ch])
        f_seen[ch] += 1

    # Count occurrence rank in L, compute LF[row], and save Occ checkpoints for FM-Index
    l_seen, l_rank, lf_map, occ_table = Counter(), [], [], []
    for ch in l_col:
        rank_in_l = l_seen[ch]
        l_rank.append(rank_in_l)
        lf_map.append(c_table[ch] + rank_in_l)  # <--- The core LF-Mapping formula!
        l_seen[ch] += 1
        occ_table.append({sym: l_seen[sym] for sym in alphabet})

    return {
        "n_total": len(bwt_string), "l_column": l_col, "f_column": f_col,
        "alphabet": alphabet, "counts": counts, "c_table": c_table,
        "f_rank_0based": f_rank, "l_rank_0based": l_rank,
        "lf_mapping": lf_map, "occ_table": occ_table,
    }


def bwt_decode(
    bwt_string: str, primary_index: Optional[int] = None,
    verbose: bool = True, expected_original: Optional[str] = None
) -> str:
    """
    How Inverse BWT works:
    In any row 'r', L[r] is the letter that comes right BEFORE F[r] in the original sequence.
    So we start at the '$' row, grab the letter in L, jump to its matching row in F using LF[r],
    and repeat N times to spell the sequence backwards!
    """
    bwt, p_idx = validate_bwt_input(bwt_string, primary_index)
    data = build_lf_mapping_structures(bwt)
    n, l_col, f_col = data["n_total"], data["l_column"], data["f_column"]
    c_table, f_rank, l_rank, lf_map = data["c_table"], data["f_rank_0based"], data["l_rank_0based"], data["lf_mapping"]

    # Start at the row where F has '$' (which is always LF[primary_index], i.e., row 0)
    curr_row = lf_map[p_idx]
    collected_backwards, trace = [], []

    for step in range(1, n + 1):
        char_before = l_col[curr_row]
        if char_before != SENTINEL:
            collected_backwards.append(char_before)

        rev_so_far = "".join(collected_backwards)
        trace.append((
            step, curr_row, f"{f_col[curr_row]}{f_rank[curr_row]+1}",
            f"{char_before}{l_rank[curr_row]+1}", lf_map[curr_row],
            rev_so_far if char_before != SENTINEL else rev_so_far + " ($)",
            rev_so_far[::-1]
        ))
        curr_row = lf_map[curr_row]  # Jump to the next row!

    # Flip the backwards letters to get the original sequence
    decoded = "".join(reversed(collected_backwards))

    if verbose:
        print(f"\n{'='*75}\nINVERSE BWT DECODING | BWT: {bwt} | Primary Index: {p_idx} | F: {''.join(f_col)}\n{'='*75}")
        print("STEP 1: C-Table (Starting row of each character in column F)")
        print(f"  {'Char':<6} | {'ASCII':<6} | {'Freq':<6} | {'C[c] (Start in F)':<18} | {'F Row Range'}")
        print("  " + "-" * 58)
        for ch in data["alphabet"]:
            start = c_table[ch]
            print(f"  {ch:^6} | {ord(ch):^6} | {data['counts'][ch]:^6} | {start:^18} | [{start} .. {start + data['counts'][ch] - 1}]")

        print("\nSTEP 2: LF Mapping Table (LF[i] = C[L[i]] + L_Rank_0)")
        print(f"  {'Row (0/1-b)':<12} | {'F_Rank':<8} | {'L_Rank (0/1-b)':<14} | {'C[L[i]] + r0':<14} | {'LF[i]'}")
        print("  " + "-" * 62)
        for i in range(n):
            ch = l_col[i]
            print(f"  {f'{i} / {i+1}':<12} | {f'{f_col[i]}_{f_rank[i]+1}':^8} | {f'{l_rank[i]} ({ch}_{l_rank[i]+1})':^14} | {f'{c_table[ch]} + {l_rank[i]}':^14} | {lf_map[i]:^5}")

        print("\nSTEP 3: Backward Reconstruction Trace (Walking right-to-left through the text)")
        print(f"  {'Step':<4} | {'Row':<4} | {'F[curr]':<7} | {'L[curr]':<7} | {'Next LF':<7} | {'Collected (Rev)':<16} | {'Suffix So Far'}")
        print("  " + "-" * 72)
        for step, r, fc, lc, nxt, rev_s, fwd_s in trace:
            print(f"  {step:<4} | {r:^4} | {fc:^7} | {lc:^7} | {nxt:^7} | {rev_s:<16} | {fwd_s}")

        status = f" | Match: {'PASS' if decoded == expected_original else 'FAIL'}" if expected_original else ""
        print(f"\nSTEP 4: Reconstructed Sequence: {decoded}{status}\n" + "=" * 75)

    return decoded

# 4. BIOINFORMATICS STATS & RUN-LENGTH CLUSTERING

def count_runs(text: str) -> Tuple[int, List[Tuple[str, int]]]:
    """Counts consecutive identical letter runs (e.g. 'AAAB' -> [('A', 3), ('B', 1)])."""
    if not text:
        return 0, []
    runs = [[text[0], 1]]
    for ch in text[1:]:
        if ch == runs[-1][0]:
            runs[-1][1] += 1  # Same letter as before, increase count
        else:
            runs.append([ch, 1])  # New letter, start a new run
    rle = [(ch, count) for ch, count in runs]
    return len(rle), rle


def find_repetitive_patterns(sequence: str) -> Dict[str, object]:
    """Finds homopolymers (like 'AAAA') and repeated short words (k-mers of length 2, 3, 4)."""
    # 1. Find homopolymers (same letter repeated 2 or more times in a row)
    _, rle = count_runs(sequence)
    homopolymers, pos = [], 0
    for ch, length in rle:
        if length >= 2:
            homopolymers.append((ch * length, length, pos))
        pos += length

    # 2. Find repeated k-mers (substrings of length k=2, 3, 4 that appear >= 2 times)
    repeated_kmers = {}
    for k in (2, 3, 4):
        kmer_counts = Counter(sequence[i:i + k] for i in range(len(sequence) - k + 1))
        repeats = {kmer: cnt for kmer, cnt in kmer_counts.items() if cnt >= 2}
        if repeats:
            repeated_kmers[k] = dict(sorted(repeats.items(), key=lambda x: (-x[1], x[0])))

    return {"homopolymers": homopolymers, "repeated_kmers": repeated_kmers}


def analyze_biological_sequence(sequence: str, seq_type: str = "AUTO") -> None:
    """Calculates GC/AT content, amino acid percentages, and compares RLE runs before/after BWT."""
    seq, dtype = validate_raw_sequence(sequence, seq_type)
    n, counts = len(seq), Counter(seq)
    bwt_str, p_idx = bwt_encode(seq, verbose=False)
    orig_r, orig_rle = count_runs(seq)
    bwt_r, bwt_rle = count_runs(bwt_str)

    print(f"\n{'='*75}\nBIOINFORMATICS & BWT CLUSTERING REPORT | {seq} ({dtype}, len={n})\n{'='*75}")
    print(f"  BWT String: {bwt_str} (Primary Index: {p_idx})")

    if dtype in ("DNA", "RNA"):
        t_or_u = "T" if dtype == "DNA" else "U"
        names = {"A": "Adenine", "T": "Thymine", "U": "Uracil", "G": "Guanine", "C": "Cytosine", "N": "Ambiguous"}
        print(f"\n[A] {dtype} COMPOSITION:")
        for base in ["A", t_or_u, "G", "C", "N"]:
            if base in counts or base != "N":
                c = counts.get(base, 0)
                print(f"  {base} ({names[base]:<9}): {c:>3} ({c / n * 100:5.2f}%)")
        gc = (counts["G"] + counts["C"]) / n * 100
        at = (counts["A"] + counts[t_or_u]) / n * 100
        print(f"  GC Content: {gc:.2f}% | A{t_or_u} Content: {at:.2f}%")
        print("  Bio Note: Aligners (Bowtie2/BWA/HISAT2) use BWT+FM-Index to map reads in O(M) time.")

    elif dtype == "PROTEIN":
        print("\n[A] PROTEIN AMINO ACID COMPOSITION:")
        for aa, c in sorted(counts.items()):
            print(f"  {aa} ({AA_NAMES.get(aa, 'Unknown'):<14}): {c:>3} ({c / n * 100:5.2f}%)")
    else:
        print("\n[A] TEXT CHARACTER FREQUENCIES:")
        for ch, c in sorted(counts.items()):
            print(f"  '{ch}' (ASCII {ord(ch):>3}): {c:>3} ({c / n * 100:5.2f}%)")

    rep = find_repetitive_patterns(seq)
    print("\n[B] REPEATS & RUN-LENGTH CLUSTERING (RLE):")
    hp = ", ".join(f"'{s}' (len={l} @{p})" for s, l, p in rep["homopolymers"]) or "None"
    print(f"  Homopolymers   : {hp}")
    if rep["repeated_kmers"]:
        for k, kd in rep["repeated_kmers"].items():
            print(f"  Repeated {k}-mers: " + ", ".join(f"'{km}' (x{c})" for km, c in kd.items()))
    else:
        print("  Repeated k-mers: None")

    print(f"  Original Runs  : {orig_r} -> {' '.join(f'{c}{k}' for c, k in orig_rle)}")
    print(f"  BWT Runs       : {bwt_r} -> {' '.join(f'{c}{k}' for c, k in bwt_rle)}")
    print("=" * 75)

# 5. FM-INDEX EXACT PATTERN MATCHING (BACKWARD SEARCH)

def fm_index_backward_search(sequence: str, pattern: str, verbose: bool = True) -> List[int]:
    """
    Finds where a pattern appears in the sequence by searching BACKWARDS (right-to-left).
    At each letter 'c', we narrow down the matching row range [start_row .. end_row] using:
      new_start = C[c] + Occ(c, start_row - 1)
      new_end   = C[c] + Occ(c, end_row) - 1
    """
    seq, _ = validate_raw_sequence(sequence, "AUTO")
    pat = pattern.strip().upper() if seq.isupper() else pattern.strip()
    if not pat:
        raise ValueError("Search pattern cannot be empty.")

    seq_t = seq + SENTINEL
    n = len(seq_t)
    sorted_rot = sorted([(i, seq_t[i:] + seq_t[:i]) for i in range(n)], key=lambda x: _sentinel_sort_key(x[1]))
    suffix_array = [i for i, _ in sorted_rot]
    bwt = "".join(r[-1] for _, r in sorted_rot)

    data = build_lf_mapping_structures(bwt)
    c_table, occ_table, alphabet = data["c_table"], data["occ_table"], data["alphabet"]

    def occ(ch: str, row_idx: int) -> int:
        """Returns how many times 'ch' appears in L[0 .. row_idx]."""
        return 0 if row_idx < 0 or ch not in c_table else occ_table[row_idx].get(ch, 0)

    start_row, end_row, matched, trace = 0, n - 1, True, []
    for step, ch in enumerate(reversed(pat), 1):
        if ch not in c_table:
            matched = False
            trace.append((step, ch, start_row, end_row, -1, -2, "Letter not in sequence"))
            break
        new_start = c_table[ch] + occ(ch, start_row - 1)
        new_end = c_table[ch] + occ(ch, end_row) - 1
        if new_start > new_end:
            matched = False
            trace.append((step, ch, start_row, end_row, new_start, new_end, "No match (range empty)"))
            break
        trace.append((step, ch, start_row, end_row, new_start, new_end, f"{new_end - new_start + 1} match(es)"))
        start_row, end_row = new_start, new_end

    positions = sorted(suffix_array[start_row:end_row + 1]) if matched else []

    if verbose:
        print(f"\n{'='*75}\nFM-INDEX BACKWARD SEARCH | Ref: {seq} | Pattern: {pat}\n{'='*75}")
        hdr = " | ".join(f"{s:^3}" for s in alphabet)
        print(f"  Row | L[i] | SA[i] | {hdr}  (Occ Table: count of char in L[0..i])")
        print("  " + "-" * (22 + 6 * len(alphabet)))
        for i in range(n):
            vals = " | ".join(f"{occ_table[i][s]:^3}" for s in alphabet)
            print(f"  {i:<3} | {bwt[i]:^4} | {suffix_array[i]:^5} | {vals}")

        print("\n  Backward Search Steps:")
        for s_no, ch, osp, oep, nsp, nep, note in trace:
            print(f"  Step {s_no}: char='{ch}' | [{osp}..{oep}] -> [{nsp}..{nep}] ({note})")

        if matched:
            print(f"\n  MATCH FOUND! 0-based Pos: {positions} | 1-based Pos: {[p+1 for p in positions]}")
        else:
            print(f"\n  Pattern '{pat}' NOT found.")
        print("=" * 75)

    return positions


# 6. QUICK SUMMARY, TEST CASES & INTERACTIVE MENU

def display_algorithm_explanation() -> None:
    print(f"\n{'='*75}\nQUICK CHEAT SHEET: BWT, LF-MAPPING & FM-INDEX\n{'='*75}")
    print("  * BWT      : Appends '$', sorts cyclic shifts, extracts Last column (L) to cluster repeats.")
    print("  * LF-Map   : k-th 'c' in L is the same text char as k-th 'c' in F -> LF[i] = C[L[i]] + rank(L[i]).")
    print("  * SA vs FM : Suffix Array stores sorted suffix positions; FM-Index = BWT + C + Occ + sampled SA.")
    print("  * Bio Note : BWT compresses/indexes genomes; full aligners add dynamic programming for gaps.")
    print("=" * 75)


def run_predefined_test_cases(detailed: bool = False) -> None:
    cases = [
        ("1. Simple DNA", "GATTACA"), ("2. Repetitive DNA", "AAAAAA"),
        ("3. Diverse DNA", "ATGCGATC"), ("4. RNA Sequence", "AUGCUAGC"),
        ("5. Protein Seq", "MKWVTFISLL"), ("6. Ordinary Text", "BANANA"),
    ]
    print(f"\n{'='*80}\n{'Test Case':<20} | {'Original':<11} | {'BWT':<12} | {'Idx (0/1b)':<10} | {'Decoded':<11} | {'Status'}\n{'-'*80}")
    for title, seq in cases:
        bwt, idx = bwt_encode(seq, verbose=detailed)
        dec = bwt_decode(bwt, idx, verbose=detailed, expected_original=seq)
        print(f"{title:<20} | {seq:<11} | {bwt:<12} | {f'{idx}/{idx+1}':^10} | {dec:<11} | {'PASS' if dec == seq else 'FAIL'}")
    print("=" * 80)


def interactive_menu() -> None:
    while True:
        print("\n=== BWT BIOINFORMATICS TOOL ===")
        print("1. Encode (BWT)  2. Decode (IBWT)  3. Encode & Decode  4. Bio Stats  5. Theory & FM-Demo  6. Run 6 Tests  7. Exit")
        try:
            choice = input("Choice (1-7): ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        try:
            if choice == "1":
                bwt_encode(input("Enter sequence: "), verbose=True)
            elif choice == "2":
                bwt_in = input(f"Enter BWT string (with '{SENTINEL}'): ").strip()
                idx_in = input("Enter 0-based Primary Index (Enter for auto): ").strip()
                bwt_decode(bwt_in, int(idx_in) if idx_in else None, verbose=True)
            elif choice == "3":
                seq, _ = validate_raw_sequence(input("Enter sequence: "), "AUTO")
                bwt, idx = bwt_encode(seq, verbose=True)
                dec = bwt_decode(bwt, idx, verbose=True, expected_original=seq)
                print(f"\n[Round-Trip]: {seq} -> {bwt} (idx {idx}) -> {dec} | {'SUCCESS' if seq == dec else 'FAIL'}")
            elif choice == "4":
                seq = input("Enter sequence: ")
                t = input("Type (AUTO/DNA/RNA/PROTEIN/TEXT) [AUTO]: ").strip() or "AUTO"
                analyze_biological_sequence(seq, t)
            elif choice == "5":
                display_algorithm_explanation()
                if input("Run FM-Index Backward Search demo? (y/n) [y]: ").strip().lower() in ("", "y", "yes"):
                    ref = input("Reference [GATTACA]: ").strip() or "GATTACA"
                    pat = input("Pattern [ATT]: ").strip() or "ATT"
                    fm_index_backward_search(ref, pat, verbose=True)
            elif choice == "6":
                det = input("Show full tables for all 6 tests? (y/n) [n]: ").strip().lower() in ("y", "yes")
                run_predefined_test_cases(detailed=det)
            elif choice == "7":
                print("Goodbye!")
                break
            else:
                print("Invalid choice. Pick 1-7.")
        except ValueError as err:
            print(f"[Error]: {err}")


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "--test":
        run_predefined_test_cases(detailed=True)
    else:
        interactive_menu()


=== BWT BIOINFORMATICS TOOL ===
1. Encode (BWT)  2. Decode (IBWT)  3. Encode & Decode  4. Bio Stats  5. Theory & FM-Demo  6. Run 6 Tests  7. Exit
Choice (1-7): 3
Enter sequence: BANANA

FORWARD BWT ENCODING | Input: BANANA (PROTEIN) | T: BANANA$
STEP 1: All Cyclic Rotations (Moving letters in a circle)
  Shift (0/1-b)  | Rotation        | First | Last
  -----------------------------------------------
  0 / 1          | BANANA$         |   B   |  $   <-- Original
  1 / 2          | ANANA$B         |   A   |  B  
  2 / 3          | NANA$BA         |   N   |  A  
  3 / 4          | ANA$BAN         |   A   |  N  
  4 / 5          | NA$BANA         |   N   |  A  
  5 / 6          | A$BANAN         |   A   |  N  
  6 / 7          | $BANANA         |   $   |  A  

STEP 2: Sorted BWT Matrix (Sorting puts identical contexts next to each other)
  Row (0/1-b)  | SA[i] | F   | Sorted Rotation | L (BWT)
  -----------------------------------------------------
  0 / 1        | 6     |  $  | $BANANA 